In [ ]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/

In [ ]:
import pandas as pd
import numpy as np
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, TVAESynthesizer
from sdv.metadata import SingleTableMetadata

import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPUs detected: {physical_devices}")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPUs detected. TensorFlow will run on CPU.")

FILE_PATH = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.zip"
NUM_SYNTHETIC_SAMPLES = 5
EPOCHS = 200

print(f"Attempting to load file: {FILE_PATH}")
df = pd.read_csv(FILE_PATH, compression='zip', low_memory=False)

df.columns = df.columns.str.strip()

print(f"Original data shape: {df.shape}")

print("Using the full dataset for training.")

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.replace([np.inf, -np.inf], np.nan, inplace=True)

df.fillna(df.mean(numeric_only=True), inplace=True)

print(f"Cleaned data shape (after filling NaNs): {df.shape}")

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)

if 'Label' in df.columns:
    metadata.update_column('Label', sdtype='categorical')

synthesizers = {
    "CTGAN": CTGANSynthesizer(metadata, epochs=EPOCHS, verbose=True),
    "CopulaGAN": CopulaGANSynthesizer(metadata, epochs=EPOCHS, verbose=True),
    "TVAE": TVAESynthesizer(metadata, epochs=EPOCHS)
}

synthetic_datasets = {}

for name, synthesizer in synthesizers.items():
    print(f"\n--- Training {name} model ---")
    try:
        synthesizer.fit(df)
        print(f"{name} training complete.")

        print(f"Generating {NUM_SYNTHETIC_SAMPLES} synthetic samples with {name}...")
        samples = synthesizer.sample(NUM_SYNTHETIC_SAMPLES)
        synthetic_datasets[name] = samples
        print(f"{name} Synthetic Samples:\n", samples)
    except Exception as e:
        print(f"Error during {name} training or sampling: {e}")

print("\n--- Summary of Generated Synthetic Data ---")
for name, data in synthetic_datasets.items():
    print(f"\n{name} Synthetic Data (first 5 rows):")
    print(data.head())
    print(f"{name} Data Shape: {data.shape}")

No GPUs detected. TensorFlow will run on CPU.
Attempting to load file: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.zip
Original data shape: (225745, 79)
Using the full dataset for training.
Cleaned data shape (after filling NaNs): (225745, 79)


/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(



--- Training CTGAN model ---


KeyboardInterrupt: 

In [ ]:
df.columns

Index(['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes',
       'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt',
       'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt',
       'synack', 'ackdat', 'smean', 'dmean', 'trans_depth',
       'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports',
       'attack_cat', 'label'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPUs detected: {physical_devices}")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPUs detected. TensorFlow will run on CPU.")

file_path = "/content/UNSW_NB15_training-set.parquet"
print(f"Attempting to load file: {file_path}")
df = pd.read_parquet(file_path)
print(f"Original data shape: {df.shape}")

print(f"Using the full dataset for training. Current shape: {df.shape}")

discrete_columns = ['proto', 'service', 'state', 'attack_cat', 'label']
continuous_columns = [col for col in df.columns if col not in discrete_columns]

df[continuous_columns] = df[continuous_columns].apply(pd.to_numeric, errors='coerce')
df[continuous_columns] = df[continuous_columns].replace([np.inf, -np.inf], np.nan)
df[continuous_columns] = df[continuous_columns].fillna(df[continuous_columns].mean())

for col in discrete_columns:
    df[col] = df[col].astype(object).fillna('unknown')

print(f"Cleaned data shape (after filling NaNs): {df.shape}")
print("\nFirst 5 rows of cleaned data:")
print(df.head())

metadata = Metadata.detect_from_dataframe(df)
for col in discrete_columns:
    metadata.update_column(col, sdtype='categorical')

EPOCHS = 200

synthesizers = {
    "CTGAN": CTGANSynthesizer(metadata, epochs=EPOCHS, verbose=True),
    "CopulaGAN": CopulaGANSynthesizer(metadata, epochs=EPOCHS, verbose=True),
    "TVAE": TVAESynthesizer(metadata, epochs=EPOCHS, verbose=True)
}

synthetic_data = {}

for name, synth in synthesizers.items():
    print(f"\n--- Training {name} ---")
    synth.fit(df)
    print(f"{name} training complete.")

    print(f"\nGenerating 10 synthetic samples using {name}...")
    samples = synth.sample(10)
    synthetic_data[name] = samples
    print(f"\n{name} Synthetic Samples:\n", samples)

print("\n--- Summary of Generated Synthetic Data ---")
for name, data in synthetic_data.items():
    print(f"\n{name} Data (first 5 rows):")
    print(data.head())
    print(f"{name} Shape: {data.shape}")

Attempting to load file: /content/UNSW_NB15_training-set.parquet
Original data shape: (175341, 36)
Sampled 10000 rows for training.
Cleaned data shape (after filling NaNs): (10000, 36)

First 5 rows of cleaned data:
        dur proto service state  spkts  dpkts  sbytes   dbytes           rate  \
0  2.736664   tcp       -   FIN    232    438   13350   548216     244.458206   
1  0.000009   udp     dns   INT      2      0     114        0  111111.109375   
2  5.788526   tcp       -   FIN     36     34    6102     3892      11.920133   
3  3.849634   tcp       -   FIN    448    858   25160  1094788     338.993286   
4  0.001052   udp     dns   CON      2      2     130      162    2851.711182   

          sload  ...  trans_depth  response_body_len  ct_src_dport_ltm  \
0  3.885899e+04  ...            0                  0                 2   
1  5.066666e+07  ...            0                  0                10   
2  8.199669e+03  ...            0                  0                 1   
3

/tmp/ipython-input-3-2484434566.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].astype(object).fillna('unknown')
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-0.79) | Discrim. (-0.15): 100%|██████████| 20/20 [01:42<00:00,  5.10s/it]


CTGAN training complete.

Generating 10 synthetic samples using CTGAN...

CTGAN Synthetic Samples:
         dur proto service state  spkts  dpkts  sbytes  dbytes            rate  \
0  0.098490  unas       -   FIN      3    241    5151     590   327357.000000   
1  0.063940   tcp     dns   INT     12     34      46    1001   101368.718750   
2  0.096476   tcp       -   INT     11     28    5080       0        0.000000   
3  1.918233   udp    smtp   INT      8     40      46       0   105224.539062   
4  0.077070   tcp     dns   INT    371      0      46       0     6821.415039   
5  7.800488   tcp    http   CON     10     30      46       0      499.499237   
6  1.813110   udp       -   INT     60     39    3309       0  1000000.000000   
7  1.288543  ifmp    http   INT      5      8      46       0        0.000000   
8  0.322024   tcp       -   FIN     13     14      46       0        0.000000   
9  0.041226   pup    http   INT      7     11    3798       0   122193.664062   

        

Gen. (-0.06) | Discrim. (-1.18): 100%|██████████| 20/20 [01:42<00:00,  5.13s/it]


CopulaGAN training complete.

Generating 10 synthetic samples using CopulaGAN...

CopulaGAN Synthetic Samples:
             dur proto service state  spkts  dpkts  sbytes  dbytes  \
0  1.139162e-05   tcp     ssh   FIN     18      7     979       0   
1  3.461607e-04   tcp     dns   INT      2      0    1321       0   
2  1.040088e-06   tcp       -   FIN      5      0     284     372   
3  2.224725e-06   udp    smtp   FIN      3      0     384       0   
4  6.860018e-03   tcp    pop3   INT      2     95      64   11215   
5  2.309746e-05   tcp    http   CON     33      3     134       0   
6  1.005147e-02   udp       -   FIN      2      0      54       0   
7  4.798014e-01  ifmp       -   INT      2      5      86    1288   
8  1.005888e-26   tcp       -   FIN      2      0     238       0   
9  5.839777e-06   pup    http   INT      8     10     525    1485   

             rate         sload  ...  trans_depth  response_body_len  \
0     8265.255859  2.570542e+03  ...            1       

Loss: -32.136: 100%|██████████| 20/20 [00:21<00:00,  1.09s/it]


TVAE training complete.

Generating 10 synthetic samples using TVAE...

TVAE Synthetic Samples:
         dur proto service state  spkts  dpkts  sbytes  dbytes           rate  \
0  0.537591   tcp       -   FIN      7      5    2920     444     161.358765   
1  0.064574   tcp       -   INT     11      1     628       0       0.000000   
2  0.000000   tcp       -   INT      2      6      46     899       0.000000   
3  0.963636   tcp    http   FIN     36     20     671    2950       0.000000   
4  0.629414   tcp       -   FIN     12      0      46     857    2199.581299   
5  0.042982  unas       -   INT      3      0    1761       0  113514.523438   
6  0.015249   udp       -   INT      2      4    1156       0    2561.053467   
7  0.038371   tcp       -   INT      3      3      46     377       0.000000   
8  0.055844   udp       -   INT      6      2      46    1037    4239.990723   
9  0.753906   tcp       -   FIN      8      2     615     632    3425.900146   

          sload  ...  

In [ ]:
import pandas as pd
import numpy as np
import os
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPUs detected: {physical_devices}")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPUs detected. TensorFlow will run on CPU.")

FILE_PATH = "/content/KDDTrain+.txt.zip"
NUM_SYNTHETIC_SAMPLES = 8
EPOCHS = 200

columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot',
    'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count',
    'serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count','dst_host_same_srv_rate',
    'dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'outcome','level'
]

if not os.path.exists(FILE_PATH):
    print(f"Error: The file '{FILE_PATH}' was not found. Please upload it to Colab.")
else:
    print(f"File '{FILE_PATH}' found. Proceeding with data loading.")

    df = pd.read_csv(FILE_PATH, header=None, names=columns)
    print(f"Original data shape: {df.shape}")

    print(f"Using the full dataset for training. Current shape: {df.shape}")

    discrete_columns = ['protocol_type', 'service', 'flag', 'outcome', 'level']
    continuous_columns = [col for col in df.columns if col not in discrete_columns]

    for col in continuous_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df[continuous_columns] = df[continuous_columns].replace([np.inf, -np.inf], np.nan)

    df[continuous_columns] = df[continuous_columns].fillna(df[continuous_columns].mean(numeric_only=True))

    for col in discrete_columns:
        df[col] = df[col].astype(str).fillna('unknown')

    print(f"Cleaned data shape (after preprocessing): {df.shape}")
    print("\nFirst 5 rows of cleaned data:")
    print(df.head())

    metadata = Metadata.detect_from_dataframe(df)
    for col in discrete_columns:
        metadata.update_column(col, sdtype='categorical')

    synthesizers = {
        "CTGAN": CTGANSynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True),
        "CopulaGAN": CopulaGANSynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True),
        "TVAE": TVAESynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True)
    }

    synthetic_datasets = {}

    for name, synth in synthesizers.items():
        print(f"\n--- Training {name} model ---")
        try:
            synth.fit(df)
            print(f"{name} training complete.")

            print(f"Generating {NUM_SYNTHETIC_SAMPLES} synthetic samples with {name}...")
            samples = synth.sample(NUM_SYNTHETIC_SAMPLES)
            synthetic_datasets[name] = samples
            print(f"{name} Synthetic Samples:\n", samples)
        except Exception as e:
            print(f"Error during {name} training or sampling: {e}")

    print("\n--- Summary of Generated Synthetic Data ---")
    for name, data in synthetic_datasets.items():
        print(f"\n{name} Data (first 5 rows):")
        print(data.head())
        print(f"{name} Shape: {data.shape}")

File '/content/KDDTrain+.txt.zip' found. Proceeding with data loading.
Original data shape: (125973, 43)
Reduced to 7500 rows for training.
Cleaned data shape (after preprocessing): (7500, 43)

First 5 rows of cleaned data:
   duration protocol_type   service  flag  src_bytes  dst_bytes  land  \
0         0           udp  domain_u    SF         36          0     0   
1         0           tcp      http    S0          0          0     0   
2         0           tcp     pop_3    S0          0          0     0   
3         0           tcp   private   REJ          0          0     0   
4         0           tcp   private  RSTR          0          0     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    1.00   
1               0       0    0  ...                    0.17   
2               0       0    0  ...                    0.08   
3               0       0    0  ...                    0.11   
4               0     

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(



--- Training CTGAN model ---


Gen. (-0.97) | Discrim. (-0.42): 100%|██████████| 15/15 [00:51<00:00,  3.45s/it]


CTGAN training complete.
Generating 8 synthetic samples with CTGAN...
CTGAN Synthetic Samples:
    duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp    telnet   SF          0        131     0   
1         0           udp    telnet   SF       3906          0     0   
2         0           tcp      http   SF       1009      36127     0   
3         0           tcp      http   SF          0        798     0   
4         0          icmp      nntp   SF       4724         92     0   
5         0           tcp  http_443   SF       3679          0     0   
6         2           tcp      http   SF       4760       6638     0   
7         0           tcp      http   SF       2586          0     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    1.00   
1               0       0    0  ...                    1.00   
2               0       0    0  ...                    0.05   
3  

Gen. (-0.44) | Discrim. (0.18): 100%|██████████| 15/15 [00:48<00:00,  3.25s/it]


CopulaGAN training complete.
Generating 8 synthetic samples with CopulaGAN...
CopulaGAN Synthetic Samples:
    duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp    telnet   SF          0      23715     0   
1         0           udp    telnet   SF          0          0     0   
2         0           tcp     pop_3   SF          0          0     0   
3         0           tcp      http   SF          0          0     0   
4         0          icmp      nntp   SF          0          0     0   
5         0           tcp  http_443   S0          0          0     0   
6         0           tcp      http   SF          0          0     0   
7         0           tcp  domain_u   SF          0      67916     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    1.00   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                   

Loss: -43.485: 100%|██████████| 15/15 [00:12<00:00,  1.19it/s]


TVAE training complete.
Generating 8 synthetic samples with TVAE...
TVAE Synthetic Samples:
    duration protocol_type  service flag  src_bytes  dst_bytes  land  \
0         0           tcp  private   S0       1053          0     0   
1         5           tcp     http   SF        842        130     0   
2         0           tcp     http   SF        415          0     0   
3         0           tcp     http   SF       3243          0     0   
4         0           tcp  private   S0          0        457     0   
5         5           tcp     http   SF        176          0     0   
6         0           tcp     http   SF       1131       1266     0   
7         4           tcp     http   SF        872        624     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.04   
1               0       0    0  ...                    1.00   
2               0       0    0  ...                    0.99   
3              

In [ ]:
import pandas as pd
import numpy as np
import os
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"GPUs detected: {physical_devices}")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPUs detected. TensorFlow will run on CPU.")

FILE_PATH = "/content/reduced_data_1.csv.zip"
NUM_SYNTHETIC_SAMPLES = 5
EPOCHS = 200

if not os.path.exists(FILE_PATH):
    print(f"Error: The file '{FILE_PATH}' was not found. Please upload it to Colab.")
else:
    print(f"File '{FILE_PATH}' found. Proceeding with data loading.")

    df = pd.read_csv(FILE_PATH, low_memory=False)
    print(f"Original data shape: {df.shape}")

    print(f"Using the full dataset for training. Current shape: {df.shape}")

    discrete_columns = [
        'flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport',
        'state', 'attack', 'category', 'subcategory'
    ]

    continuous_columns = [col for col in df.columns if col not in discrete_columns]

    for col in continuous_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df[continuous_columns] = df[continuous_columns].replace([np.inf, -np.inf], np.nan)

    df[continuous_columns] = df[continuous_columns].fillna(df[continuous_columns].mean())

    for col in discrete_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).fillna('unknown')
        else:
            print(f"Warning: Discrete column '{col}' not found in DataFrame.")

    initial_rows = df.shape[0]
    df_clean = df.dropna().reset_index(drop=True)
    if df_clean.shape[0] < initial_rows:
        print(f"Warning: {initial_rows - df_clean.shape[0]} rows dropped due to remaining NaNs.")

    print(f"Cleaned data shape (after preprocessing and final NaN check): {df_clean.shape}")
    print("\nFirst 5 rows of cleaned data:")
    print(df_clean.head())

    if df_clean.shape[0] == 0:
        print("Error: Cleaned dataframe is empty after processing. Cannot train any synthesizers.")
    else:
        metadata = Metadata.detect_from_dataframe(df_clean)
        for col in discrete_columns:
            if col in df_clean.columns:
                metadata.update_column(col, sdtype='categorical')

        synthesizers = {
            "CTGAN": CTGANSynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True),
            "CopulaGAN": CopulaGANSynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True),
            "TVAE": TVAESynthesizer(metadata=metadata, epochs=EPOCHS, verbose=True)
        }

        synthetic_datasets = {}

        for name, synth in synthesizers.items():
            print(f"\n--- Training {name} model ---")
            try:
                synth.fit(df_clean)
                print(f"{name} training complete.")

                print(f"Generating {NUM_SYNTHETIC_SAMPLES} synthetic samples with {name}...")
                samples = synth.sample(NUM_SYNTHETIC_SAMPLES)
                synthetic_datasets[name] = samples
                print(f"{name} Synthetic Samples:\n", samples)
            except Exception as e:
                print(f"Error during {name} training or sampling: {e}")

        print("\n--- Summary of Generated Synthetic Data ---")
        for name, data in synthetic_datasets.items():
            print(f"\n{name} Data (first 5 rows):")
            print(data.head())
            print(f"{name} Shape: {data.shape}")

File '/content/reduced_data_1.csv.zip' found. Proceeding with data loading.
Original data shape: (1000000, 46)
Reduced to 10000 rows for training.
Cleaned data shape (after preprocessing and final NaN check): (5000, 46)

First 5 rows of cleaned data:
   pkSeqID         stime flgs  flgs_number proto  proto_number  \
0   987232  1.528085e+09    e            1   udp             3   
1    79955  1.528081e+09  e s            2   tcp             1   
2   567131  1.528081e+09  e s            2   tcp             1   
3   500892  1.528081e+09    e            1   tcp             1   
4    55400  1.528081e+09  e s            2   tcp             1   

             saddr  sport          daddr dport  ...  AR_P_Proto_P_DstIP  \
0  192.168.100.148  34168  192.168.100.6    80  ...            0.319763   
1  192.168.100.148  25759  192.168.100.6    80  ...            0.201667   
2  192.168.100.149  48314  192.168.100.5    80  ...            0.076042   
3  192.168.100.150   5356  192.168.100.3    80  ... 

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:128: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(



--- Training CTGAN model ---
PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
stime                  11
flgs                   4
flgs_number            4
proto                  2
proto_number           2
saddr                  6
sport                  4841
daddr                  6
dport                  3
pkts                   11
bytes                  11
state                  3
ltime                  11
seq                    11
dur                    11
mean                   11
stddev                 11
sum                    11
min                    11
max                    11
spkts                  11
dpkts                  5
sbytes                 11
dbytes                 9
rate                   11
srate                  11
drate                  11
TnBPSrcIP              11
TnBPDstIP              11
TnP_PSrcIP             11
TnP_

Gen. (1.13) | Discrim. (0.04): 100%|██████████| 5/5 [06:16<00:00, 75.31s/it]


CTGAN training complete.
Generating 5 synthetic samples with CTGAN...
CTGAN Synthetic Samples:
     pkSeqID         stime flgs  flgs_number proto  proto_number  \
0  12990897  1.528081e+09  e g            1   tcp             1   
1   1388625  1.528081e+09  e g            5   tcp             1   
2  12045166  1.528081e+09  e s            2   tcp             1   
3   9086062  1.528081e+09    e            2   tcp             1   
4   9987125  1.528081e+09    e            1   udp             1   

             saddr  sport          daddr dport  ...  AR_P_Proto_P_DstIP  \
0  192.168.100.150  11983  192.168.100.6    80  ...            0.179218   
1  192.168.100.148  18958  192.168.100.3    80  ...            0.179056   
2  192.168.100.148  23272  192.168.100.6    80  ...            0.059529   
3  192.168.100.150  59133  192.168.100.7    80  ...            0.124117   
4  192.168.100.149  11519  192.168.100.6    80  ...            0.171435   

   N_IN_Conn_P_DstIP N_IN_Conn_P_SrcIP AR_P_Proto_

Gen. (1.37) | Discrim. (0.18): 100%|██████████| 5/5 [06:20<00:00, 76.18s/it]


CopulaGAN training complete.
Generating 5 synthetic samples with CopulaGAN...
CopulaGAN Synthetic Samples:
     pkSeqID         stime flgs  flgs_number proto  proto_number  \
0  12990897  1.528081e+09  e g            1   udp             1   
1   1388625  1.528081e+09  e g            5   tcp             1   
2  12045166  1.528081e+09  e s            2   tcp             1   
3   9086062  1.528081e+09    e            2   tcp             1   
4   9987125  1.528081e+09    e            1   udp             1   

             saddr  sport          daddr dport  ...  AR_P_Proto_P_DstIP  \
0  192.168.100.150  11983  192.168.100.6    80  ...            0.138617   
1  192.168.100.148  18958  192.168.100.3    80  ...            0.162948   
2  192.168.100.148  23272  192.168.100.6    80  ...            5.491581   
3  192.168.100.150  59133  192.168.100.7    80  ...            0.183769   
4  192.168.100.148  34197  192.168.100.6    80  ...            0.132416   

   N_IN_Conn_P_DstIP N_IN_Conn_P_SrcIP

Loss: 53.924: 100%|██████████| 5/5 [00:20<00:00,  4.04s/it]


TVAE training complete.
Generating 5 synthetic samples with TVAE...
TVAE Synthetic Samples:
     pkSeqID         stime flgs  flgs_number proto  proto_number  \
0  12990897  1.528081e+09  e s            2   tcp             1   
1   1388625  1.528081e+09    e            2   tcp             1   
2  12045166  1.528081e+09    e            2   tcp             1   
3   9086062  1.528081e+09  e s            2   tcp             1   
4   9987125  1.528081e+09    e            2   tcp             1   

             saddr  sport          daddr dport  ...  AR_P_Proto_P_DstIP  \
0  192.168.100.147  56850  192.168.100.5    80  ...            0.241944   
1  192.168.100.147  56850  192.168.100.7    80  ...            0.266869   
2  192.168.100.147  56850  192.168.100.5    80  ...            0.382964   
3  192.168.100.148  34013  192.168.100.6    80  ...            0.298969   
4  192.168.100.147  56850  192.168.100.5    80  ...            0.158111   

   N_IN_Conn_P_DstIP N_IN_Conn_P_SrcIP AR_P_Proto_P_S